# Simio – Single-run report (one system)
This notebook loads CSV outputs produced by `simio_xtc_all_properties` for one trajectory/system and builds a compact, non-duplicated diagnostic report.

The report combines profile plots, transport diagnostics (MSD/VACF), channel occupancy summaries, and a final scalar summary table.

All figures and tables are saved under `report/` inside the current run directory.


In [ ]:

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120


## 0) Locate outputs
This notebook assumes it is executed **inside one run output directory** (the folder containing `run.log` and CSV files).

Artifacts are written to:
- `./report/figures/*.png`
- `./report/summary_table.csv`
- `./report/coverage_table.csv`


In [ ]:
RUN_DIR = Path.cwd().resolve()
LOG_PATH = RUN_DIR / "run.log"
REPORT_DIR = RUN_DIR / "report"
FIG_DIR = REPORT_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"RUN_DIR    : {RUN_DIR}")
print(f"LOG_PATH   : {LOG_PATH}")
print(f"REPORT_DIR : {REPORT_DIR}")
print(f"FIG_DIR    : {FIG_DIR}")

import re

def resolve_csv(stem: str) -> Path:
    candidates = [
        RUN_DIR / f"{stem}.csv",
        RUN_DIR / f"results_{stem}.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

def load_csv(stem: str):
    p = resolve_csv(stem)
    if not p.exists():
        return None
    df = pd.read_csv(p)
    if len(df.columns) > 0 and str(df.columns[0]).startswith("]"):
        df = df.rename(columns={df.columns[0]: str(df.columns[0]).lstrip("]")})
    return df

def savefig_and_show(name: str, fig=None):
    if fig is None:
        fig = plt.gcf()
    out = FIG_DIR / f"{name}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved figure: {out}")

def parse_log_metadata(log_text: str):
    meta = {
        "xtc_filename": None,
        "processed_frames": None,
        "cache_stats": None,
    }

    m_xtc = re.search(r"\bfile=([^\s]+)", log_text)
    if m_xtc:
        meta["xtc_filename"] = Path(m_xtc.group(1)).name

    m_frames = re.search(r"Processed\s+(\d+)\s+frame\(s\)", log_text)
    if m_frames:
        meta["processed_frames"] = int(m_frames.group(1))

    m_cache = re.search(
        r"\[simio-cache\]\s+gets=(\d+)\s+hits=(\d+)\s+puts=(\d+)\s+dup_puts=(\d+)",
        log_text,
    )
    if m_cache:
        meta["cache_stats"] = {
            "gets": int(m_cache.group(1)),
            "hits": int(m_cache.group(2)),
            "puts": int(m_cache.group(3)),
            "dup_puts": int(m_cache.group(4)),
        }

    return meta

def csv_stem_from_path(path: Path) -> str:
    stem = path.stem
    if stem.startswith("results_"):
        stem = stem[len("results_"):]
    return stem

log_text = LOG_PATH.read_text(encoding="utf-8", errors="replace") if LOG_PATH.exists() else ""
LOG_META = parse_log_metadata(log_text)
print("LOG_META:", LOG_META)

KNOWN_STEMS = [
    "density_x",
    "density_z",
    "density_z_in_x_channel",
    "dipole_x",
    "dipole_z",
    "dipole_z_in_x_channel",
    "coord_x",
    "channel_count_xz",
    "state_z_channel",
    "drift_channel",
    "gating_flux",
    "jump_msd",
    "msd_x_channel",
    "msd_z_channel",
    "vacf_y",
    "vacf_z_channel_raw",
    "vacf_x_channel",
]

files = {stem: resolve_csv(stem) for stem in KNOWN_STEMS}
RUN_CSV_FILES = sorted([p for p in RUN_DIR.glob("*.csv") if p.is_file()])

present = {k: p for k,p in files.items() if p.exists()}
missing = {k: str(p) for k,p in files.items() if not p.exists()}

print("Present known-property files:")
for k,p in present.items():
    print(f"  - {k:24s}: {p.name}")
if missing:
    print("\nMissing known-property files (skipped gracefully):")
    for k,p in missing.items():
        print(f"  - {k:24s}: {Path(p).name}")

print(f"\nAll CSV files found in RUN_DIR: {len(RUN_CSV_FILES)}")
for p in RUN_CSV_FILES:
    print(f"  - {p.name}")


## Helpers
Helper utilities provide robust CSV discovery (`<stem>.csv` or `results_<stem>.csv`), safe loading, log metadata parsing, and table/figure export.

`run.log` is parsed to recover trajectory name, processed frame count, and cache stats when available.


In [ ]:
def read_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    # Fix occasional stray characters in first header (e.g. ']lag_frames')
    if df.columns[0].startswith("]"):
        df = df.rename(columns={df.columns[0]: df.columns[0].lstrip("]")})
    return df

def tail_stats(series: pd.Series, n=10):
    tail = series.tail(n)
    return float(tail.mean()), float(tail.std()), float(tail.iloc[-1]), float(tail.iloc[0])

def linfit(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]; y = y[m]
    if len(x) < 2:
        return np.nan, np.nan, np.nan
    A = np.vstack([x, np.ones_like(x)]).T
    slope, intercept = np.linalg.lstsq(A, y, rcond=None)[0]
    # R^2
    yhat = slope*x + intercept
    ss_res = np.sum((y-yhat)**2)
    ss_tot = np.sum((y-np.mean(y))**2)
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    return float(slope), float(intercept), float(r2)

def show_summary_df(rows, title="Summary"):
    if not rows:
        print(f"{title}: no rows")
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    display(df)
    return df

def save_table(df: pd.DataFrame, filename: str):
    out = REPORT_DIR / filename
    df.to_csv(out, index=False)
    print(f"Saved table: {out}")
    return out


## 1) Density profiles along x
These profiles show species number-density variation along the channel axis (`x`) and are useful for identifying depletion/accumulation regions.

Plotted columns are detected dynamically from `density_x.csv`.


In [ ]:
if files["density_x"].exists():
    df = read_csv(files["density_x"])
    # Expect columns like: x_center_nm, rho_water_mean, rho_na_mean, rho_cl_mean (or *_nm3)
    xcol = [c for c in df.columns if c.startswith("x")][0]
    candidates = [c for c in df.columns if "rho" in c]
    print("Columns:", xcol, candidates)

    fig, ax = plt.subplots(figsize=(7.5,4.2))
    for c in candidates:
        ax.plot(df[xcol], df[c], label=c)
    ax.set_xlabel("x (nm)")
    ax.set_ylabel("number density (1/nm³)")
    ax.set_title("Density profiles")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    savefig_and_show("density_x", fig)


## 2) Dipole profiles along x
This section visualizes dipole-related observables binned along `x` to reveal orientational trends across the channel.

Plotted columns are detected dynamically from `dipole_x.csv`.


In [ ]:
if files["dipole_x"].exists():
    df = read_csv(files["dipole_x"])
    xcol = [c for c in df.columns if c.startswith("x")][0]
    dcols = [c for c in df.columns if ("dip" in c.lower() or "mu" in c.lower()) and c != xcol]
    print("Columns:", xcol, dcols)

    fig, ax = plt.subplots(figsize=(7.5,4.2))
    for c in dcols:
        ax.plot(df[xcol], df[c], label=c)
    ax.set_xlabel("x (nm)")
    ax.set_ylabel("dipole component / magnitude (a.u.)")
    ax.set_title("Dipole vs x")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    savefig_and_show("dipole_x", fig)


## 3) Coordination / H-bond style observables along x
Coordination and hydrogen-bond-like descriptors are plotted versus `x` to highlight structural changes across the channel region.

Plotted columns are taken from `coord_x.csv`.


In [ ]:
if files["coord_x"].exists():
    df = read_csv(files["coord_x"])
    xcol = [c for c in df.columns if c.startswith("x")][0]
    cols = [c for c in df.columns if c != xcol]
    print("Columns:", xcol, cols)

    fig, ax = plt.subplots(figsize=(7.5,4.2))
    for c in cols:
        ax.plot(df[xcol], df[c], label=c)
    ax.set_xlabel("x (nm)")
    ax.set_ylabel("coordination / count")
    ax.set_title("Coordination-like observables vs x")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    savefig_and_show("coord_x", fig)


## 4) Gating flux (counts + simple slope fits)
This is *counts vs time* across center/seam gating definitions. Cumulative net crossings are plotted per species.

A simple linear fit over the second half of the trajectory is reported as a quick net-rate diagnostic.


In [ ]:
if files["gating_flux"].exists():
    df = read_csv(files["gating_flux"])
    print("Columns:", list(df.columns))

    tcol = "time_ps" if "time_ps" in df.columns else ("frame_idx" if "frame_idx" in df.columns else df.columns[0])

    planes = ["center", "seam"]
    species = ["water", "na", "cl"]

    # Build derived net and cumulative-net columns
    cum_net_cols = []
    for p in planes:
        for s in species:
            l = f"{p}_{s}_left"
            r = f"{p}_{s}_right"
            if l in df.columns and r in df.columns:
                net = f"{p}_{s}_net"
                cum_net = f"{p}_{s}_cum_net"
                df[net] = df[r] - df[l]
                df[cum_net] = df[net].cumsum()
                cum_net_cols.append(cum_net)

    # Plot: first center, second seam; each has water/na/cl cumulative net
    fig, axs = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

    for i, p in enumerate(planes):
        ax = axs[i]
        for s, lbl in [("water", "Water"), ("na", "Na+"), ("cl", "Cl-")]:
            c = f"{p}_{s}_cum_net"
            if c in df.columns:
                ax.plot(df[tcol], df[c], label=lbl, lw=2)
        ax.set_title(f"{p.capitalize()} cumulative net crossings")
        ax.set_ylabel("Cum net (right - left)")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)

    axs[-1].set_xlabel(tcol)
    plt.tight_layout()
    savefig_and_show("gating_flux", fig)

    # Optional: fit slope on last half
    i0 = len(df) // 2
    rows = []
    for c in cum_net_cols:
        slope, intercept, r2 = linfit(df[tcol].iloc[i0:], df[c].iloc[i0:])
        rows.append({
            "series": c,
            "fit_window": f"[{i0}:{len(df)-1}]",
            "slope (net/ps)": slope,
            "R2": r2
        })
    if rows:
        gating_fit = show_summary_df(rows, "gating fit")
        save_table(gating_fit, "gating_flux_fit_table.csv")


## 5) MSD baselines + channel-restricted MSD
- `jump_msd`: slab-start MSD_x/y/z baseline
- `msd_x_channel`: channel-survival MSD_x
- `msd_z_channel`: channel-survival MSD_z (raw vs unwrapped)

Together these separate baseline transport from channel-restricted behavior.


In [ ]:
# Baseline MSD_y/z from jump_msd
if files["jump_msd"].exists():
    df = read_csv(files["jump_msd"])
    lag = df["lag_ps"] if "lag_ps" in df.columns else df["lag_frames"]

    # plot MSD_y and MSD_z if present
    ycols = [c for c in df.columns if c.startswith("msd_y_") and c.endswith("_mean")]
    zcols = [c for c in df.columns if c.startswith("msd_z_") and c.endswith("_mean")]

    if ycols or zcols:
        fig, axs = plt.subplots(1, 2, figsize=(12,4.2), sharex=True)
        for c in ycols:
            axs[0].plot(lag, df[c], label=c.replace("msd_y_","").replace("_nm2_mean",""), lw=2)
        axs[0].set_title("Baseline MSD_y")
        axs[0].set_xlabel("Lag (ps)")
        axs[0].set_ylabel("MSD (nm²)")
        axs[0].grid(alpha=0.3)
        axs[0].legend(fontsize=8)

        for c in zcols:
            axs[1].plot(lag, df[c], label=c.replace("msd_z_","").replace("_nm2_mean",""), lw=2)
        axs[1].set_title("Baseline MSD_z")
        axs[1].set_xlabel("Lag (ps)")
        axs[1].set_ylabel("MSD (nm²)")
        axs[1].grid(alpha=0.3)
        axs[1].legend(fontsize=8)

        plt.tight_layout()
        savefig_and_show("jump_msd", fig)

# Channel-survival MSD_x
if files["msd_x_channel"].exists():
    df = read_csv(files["msd_x_channel"])
    lag = df["lag_ps"] if "lag_ps" in df.columns else df["lag_frames"]
    cols = [c for c in df.columns if c.startswith("msd_x_channel_") and c.endswith("_mean")]
    fig, ax = plt.subplots(figsize=(7.5,4.2))
    for c in cols:
        ax.plot(lag, df[c], label=c.replace("msd_x_channel_","").replace("_nm2_mean",""), lw=2)
    ax.set_title("MSD_x (channel survival)")
    ax.set_xlabel("Lag (ps)")
    ax.set_ylabel("MSD_x (nm²)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    savefig_and_show("msd_x_channel", fig)

    # sample counts
    ncols = [c for c in df.columns if c.startswith("n_samples")]
    if ncols:
        fig, ax = plt.subplots(figsize=(7.5,4.0))
        for c in ncols:
            ax.plot(lag, df[c], label=c, lw=2)
        ax.set_title("MSD_x_channel sample counts")
        ax.set_xlabel("Lag (ps)")
        ax.set_ylabel("n_samples")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
        plt.tight_layout()
        savefig_and_show("msd_x_channel_samples", fig)

# Channel-survival MSD_z (raw vs unwrapped)
if files["msd_z_channel"].exists():
    df = read_csv(files["msd_z_channel"])
    lag = df["lag_ps"] if "lag_ps" in df.columns else df["lag_frames"]
    fig, axs = plt.subplots(1, 3, figsize=(15,4.2), sharex=True)
    for i,sp in enumerate(["water","na","cl"]):
        cu = f"msd_z_channel_unwrapped_{sp}_nm2_mean"
        cr = f"msd_z_channel_raw_{sp}_nm2_mean"
        if cu in df.columns:
            axs[i].plot(lag, df[cu], label="unwrapped", lw=2)
        if cr in df.columns:
            axs[i].plot(lag, df[cr], label="raw", lw=2, ls="--")
        axs[i].set_title(sp)
        axs[i].set_xlabel("Lag (ps)")
        axs[i].grid(alpha=0.3)
        axs[i].legend(fontsize=8)
    axs[0].set_ylabel("MSD_z (nm²)")
    plt.tight_layout()
    savefig_and_show("msd_z_channel", fig)


## 6) VACF_y (diffusive check) and D_y plateau
The `y`-component VACF and its cumulative integral are used as a diffusion consistency check.

The tail plateau of integrated `D_y` is summarized per species.


In [ ]:
if files["vacf_y"].exists():
    df = read_csv(files["vacf_y"])
    lag = df["lag_ps"] if "lag_ps" in df.columns else df["lag_frames"]

    fig, ax = plt.subplots(figsize=(7.5,4.2))
    ax.plot(lag, df["vacf_y_water_nm2_per_ps2"], label="Water", lw=2)
    ax.plot(lag, df["vacf_y_na_nm2_per_ps2"], label="Na+", lw=2)
    ax.plot(lag, df["vacf_y_cl_nm2_per_ps2"], label="Cl-", lw=2)
    ax.axhline(0, lw=1)
    ax.set_title("VACF_y")
    ax.set_xlabel("Lag (ps)")
    ax.set_ylabel("VACF (nm²/ps²)")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    savefig_and_show("vacf_y", fig)

    fig, ax = plt.subplots(figsize=(7.5,4.2))
    ax.plot(lag, df["D_y_from_vacf_water_nm2_per_ps"]*1000, label="Water", lw=2)
    ax.plot(lag, df["D_y_from_vacf_na_nm2_per_ps"]*1000, label="Na+", lw=2)
    ax.plot(lag, df["D_y_from_vacf_cl_nm2_per_ps"]*1000, label="Cl-", lw=2)
    ax.axhline(0, lw=1)
    ax.set_title("Integral VACF_y → D_y")
    ax.set_xlabel("Lag (ps)")
    ax.set_ylabel("D_y (nm²/ns)")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    savefig_and_show("vacf_y_integral", fig)

    rows=[]
    for sp in ["water","na","cl"]:
        mu, sd, last, first = tail_stats(df[f"D_y_from_vacf_{sp}_nm2_per_ps"]*1000, n=10)
        rows.append({"species": sp, "D_y plateau mean (nm²/ns)": mu, "std(last10)": sd, "D_y(last)": last})
    vacf_y_tbl = show_summary_df(rows, "D_y from VACF")
    save_table(vacf_y_tbl, "vacf_y_plateau_table.csv")


## 7) VACF_z (channel, raw): confinement check (integral → 0)
For confined directions, the VACF integral should tend toward a small value.

This section plots raw channel-`z` VACF and its cumulative integral to assess confinement behavior.


In [ ]:
if files["vacf_z_channel_raw"].exists():
    df = read_csv(files["vacf_z_channel_raw"])
    lag = df["lag_ps"] if "lag_ps" in df.columns else df["lag_frames"]

    fig, ax = plt.subplots(figsize=(7.5,4.2))
    ax.plot(lag, df["vacf_z_channel_raw_water_nm2_per_ps2"], label="Water", lw=2)
    ax.plot(lag, df["vacf_z_channel_raw_na_nm2_per_ps2"], label="Na+", lw=2)
    ax.plot(lag, df["vacf_z_channel_raw_cl_nm2_per_ps2"], label="Cl-", lw=2)
    ax.axhline(0, lw=1)
    ax.set_title("VACF_z (channel, raw)")
    ax.set_xlabel("Lag (ps)")
    ax.set_ylabel("VACF (nm²/ps²)")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    savefig_and_show("vacf_z_channel_raw", fig)

    fig, ax = plt.subplots(figsize=(7.5,4.2))
    ax.plot(lag, df["I_z_from_vacf_channel_raw_water_nm2_per_ps"]*1000, label="Water", lw=2)
    ax.plot(lag, df["I_z_from_vacf_channel_raw_na_nm2_per_ps"]*1000, label="Na+", lw=2)
    ax.plot(lag, df["I_z_from_vacf_channel_raw_cl_nm2_per_ps"]*1000, label="Cl-", lw=2)
    ax.axhline(0, lw=1)
    ax.set_title("Integral VACF_z (channel, raw) → should approach 0")
    ax.set_xlabel("Lag (ps)")
    ax.set_ylabel("Integral (nm²/ns)")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    savefig_and_show("vacf_z_channel_raw_integral", fig)

    rows=[]
    for sp in ["water","na","cl"]:
        mu, sd, last, first = tail_stats(df[f"I_z_from_vacf_channel_raw_{sp}_nm2_per_ps"]*1000, n=10)
        rows.append({"species": sp, "I_z mean(last10) (nm²/ns)": mu, "std(last10)": sd, "I_z(last)": last})
    vacf_z_tbl = show_summary_df(rows, "I_z from VACF (channel, raw)")
    save_table(vacf_z_tbl, "vacf_z_channel_raw_table.csv")


## 8) VACF_x (channel survival) + MSD cross-check summary
Channel-survival `x`-VACF and its integral are plotted and summarized.

Use alongside channel MSD curves to cross-check transport trends.


In [ ]:
if files["vacf_x_channel"].exists():
    df = read_csv(files["vacf_x_channel"])
    lag = df["lag_ps"] if "lag_ps" in df.columns else df["lag_frames"]

    fig, ax = plt.subplots(figsize=(7.5,4.2))
    ax.plot(lag, df["vacf_x_channel_water_nm2_per_ps2"], label="Water", lw=2)
    ax.plot(lag, df["vacf_x_channel_na_nm2_per_ps2"], label="Na+", lw=2)
    ax.plot(lag, df["vacf_x_channel_cl_nm2_per_ps2"], label="Cl-", lw=2)
    ax.axhline(0, lw=1)
    ax.set_title("VACF_x (channel survival)")
    ax.set_xlabel("Lag (ps)")
    ax.set_ylabel("VACF (nm²/ps²)")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    savefig_and_show("vacf_x_channel", fig)

    fig, ax = plt.subplots(figsize=(7.5,4.2))
    ax.plot(lag, df["I_x_from_vacf_channel_water_nm2_per_ps"]*1000, label="Water", lw=2)
    ax.plot(lag, df["I_x_from_vacf_channel_na_nm2_per_ps"]*1000, label="Na+", lw=2)
    ax.plot(lag, df["I_x_from_vacf_channel_cl_nm2_per_ps"]*1000, label="Cl-", lw=2)
    ax.axhline(0, lw=1)
    ax.set_title("Integral VACF_x (channel survival)")
    ax.set_xlabel("Lag (ps)")
    ax.set_ylabel("Integral (nm²/ns)")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    savefig_and_show("vacf_x_channel_integral", fig)

    rows=[]
    for sp in ["water","na","cl"]:
        mu, sd, last, first = tail_stats(df[f"I_x_from_vacf_channel_{sp}_nm2_per_ps"]*1000, n=10)
        rows.append({"species": sp, "I_x mean(last10) (nm²/ns)": mu, "std(last10)": sd, "I_x(last)": last})
    vacf_x_tbl = show_summary_df(rows, "I_x from VACF (channel)")
    save_table(vacf_x_tbl, "vacf_x_channel_table.csv")


## 9) Diffusion summary table
Collects key scalar diagnostics (VACF-derived integrals/plateaus) into one compact table for this run.


In [ ]:
rows=[]

# VACF_y plateau
if files["vacf_y"].exists():
    dfy = read_csv(files["vacf_y"])
    for sp in ["water","na","cl"]:
        mu, sd, last, _ = tail_stats(dfy[f"D_y_from_vacf_{sp}_nm2_per_ps"]*1000, n=10)
        rows.append({"observable":"D_y_from_VACF", "species":sp, "value_nm2_per_ns":mu, "std_last10":sd})

# VACF_x integral (channel)
if files["vacf_x_channel"].exists():
    dfx = read_csv(files["vacf_x_channel"])
    for sp in ["water","na","cl"]:
        mu, sd, last, _ = tail_stats(dfx[f"I_x_from_vacf_channel_{sp}_nm2_per_ps"]*1000, n=10)
        rows.append({"observable":"I_x_from_VACF_channel", "species":sp, "value_nm2_per_ns":mu, "std_last10":sd})

# VACF_z integral (channel) ~0
if files["vacf_z_channel_raw"].exists():
    dfz = read_csv(files["vacf_z_channel_raw"])
    for sp in ["water","na","cl"]:
        mu, sd, last, _ = tail_stats(dfz[f"I_z_from_vacf_channel_raw_{sp}_nm2_per_ps"]*1000, n=10)
        rows.append({"observable":"I_z_from_VACF_channel_raw", "species":sp, "value_nm2_per_ns":mu, "std_last10":sd})

summary = pd.DataFrame(rows)
display(summary)
if not summary.empty:
    save_table(summary, "summary_table.csv")
else:
    print("summary table is empty (no compatible VACF files found)")


## 10) Density profiles along z (including in-channel variant)
Plots `density_z` and, when present, `density_z_in_x_channel` using detected density columns.

This helps compare full-domain and in-channel `z` structure.


In [ ]:
for stem, fig_name, title in [
    ("density_z", "density_z", "Density(z)"),
    ("density_z_in_x_channel", "density_z_in_x_channel", "Density(z) in x-channel"),
]:
    if not files[stem].exists():
        continue

    df = read_csv(files[stem])
    print(f"{stem} columns:", df.columns.tolist())

    z_candidates = [c for c in df.columns if c.startswith("z")]
    if not z_candidates:
        print(f"Skipping {stem}: no z column found")
        continue
    zcol = "z_center_nm" if "z_center_nm" in df.columns else z_candidates[0]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    line_cols = [
        c for c in df.columns
        if c != zcol and ("rho" in c.lower()) and ("_sem" not in c.lower())
    ]

    if not line_cols:
        print(f"Skipping {stem}: no density columns found")
        plt.close(fig)
        continue

    for c in line_cols:
        ax.plot(df[zcol], df[c], label=c, lw=2)

    ax.set_xlabel("z (nm)")
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    savefig_and_show(fig_name, fig)



## 11) Dipole profiles along z (including in-channel variant)
Plots `dipole_z` and, when present, `dipole_z_in_x_channel` to inspect orientational behavior along `z`.


In [ ]:
for stem, fig_name, title in [
    ("dipole_z", "dipole_z", "Dipole(z)"),
    ("dipole_z_in_x_channel", "dipole_z_in_x_channel", "Dipole(z) in x-channel"),
]:
    if not files[stem].exists():
        continue

    df = read_csv(files[stem])
    print(f"{stem} columns:", df.columns.tolist())

    z_candidates = [c for c in df.columns if c.startswith("z")]
    if not z_candidates:
        print(f"Skipping {stem}: no z column found")
        continue
    zcol = "z_center_nm" if "z_center_nm" in df.columns else z_candidates[0]

    dip_cols = [
        c for c in df.columns
        if c != zcol and ("mu" in c.lower() or "dip" in c.lower()) and ("_sem" not in c.lower())
    ]

    if not dip_cols:
        print(f"Skipping {stem}: no dipole columns found")
        continue

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    for c in dip_cols:
        ax.plot(df[zcol], df[c], label=c, lw=2)
    ax.set_xlabel("z (nm)")
    ax.set_ylabel("Dipole component / magnitude (a.u.)")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    savefig_and_show(fig_name, fig)


## 12) Channel count XZ (heatmap or timeseries, depending on schema)
If `channel_count_xz.csv` contains explicit x/z bin indices, a 2D heatmap is generated.

If it is frame-wise count data, a timeseries plot is produced instead.


In [ ]:
if files["channel_count_xz"].exists():
    df = read_csv(files["channel_count_xz"])
    print("channel_count_xz columns:", list(df.columns))

    # Try XZ heatmap first if explicit x/z bin columns exist.
    x_col = next((c for c in ["x_bin", "x_idx", "x", "x_center_nm"] if c in df.columns), None)
    z_col = next((c for c in ["z_bin", "z_idx", "z", "z_center_nm"] if c in df.columns), None)

    if x_col is not None and z_col is not None:
        value_candidates = [
            c for c in df.columns
            if c not in {x_col, z_col, "frame_idx", "step", "time_ps"}
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        value_col = None
        preferred = [c for c in value_candidates if ("count" in c.lower() or c.lower() in {"sol", "na", "cl", "water"})]
        if preferred:
            value_col = preferred[0]
        elif value_candidates:
            value_col = value_candidates[0]

        if value_col is not None:
            piv = df.pivot_table(index=z_col, columns=x_col, values=value_col, aggfunc="mean").sort_index().sort_index(axis=1)
            fig, ax = plt.subplots(figsize=(7.2, 5.4))
            im = ax.imshow(piv.values, origin="lower", aspect="auto", cmap="viridis")
            ax.set_title(f"channel_count_xz heatmap ({value_col})")
            ax.set_xlabel(x_col)
            ax.set_ylabel(z_col)
            cbar = plt.colorbar(im, ax=ax)
            cbar.set_label(value_col)
            plt.tight_layout()
            savefig_and_show("channel_count_xz", fig)
    else:
        # Fallback schema: per-frame in-channel counts (sol/na/cl)
        tcol = next((c for c in ["time_ps", "frame_idx", "step"] if c in df.columns), df.columns[0])
        count_cols = [
            c for c in df.columns
            if c != tcol and c not in {"step", "frame_idx", "time_ps"}
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        preferred = [c for c in count_cols if c.lower() in {"sol", "na", "cl", "water"} or "count" in c.lower() or c.startswith("n_")]
        plot_cols = preferred if preferred else count_cols

        if plot_cols:
            fig, ax = plt.subplots(figsize=(8.0, 4.6))
            for c in plot_cols:
                ax.plot(df[tcol], df[c], label=c, lw=2)
            ax.set_title("channel_count_xz (timeseries)")
            ax.set_xlabel(tcol)
            ax.set_ylabel("Count")
            ax.grid(alpha=0.3)
            ax.legend(fontsize=8)
            plt.tight_layout()
            savefig_and_show("channel_count_xz", fig)


## 13) Channel particle count per frame (dedicated)
This dedicated section plots in-channel occupancy over time (per species/region when available).

Primary source is `state_z_channel.csv` with fallback to compatible `channel_count_xz.csv` schemas.


In [ ]:
state_df = None
state_source = None
if files["state_z_channel"].exists():
    state_df = read_csv(files["state_z_channel"])
    state_source = "state_z_channel"
elif files["channel_count_xz"].exists():
    state_df = read_csv(files["channel_count_xz"])
    state_source = "channel_count_xz"

if state_df is not None:
    print(f"channel particle source: {state_source}")
    print("columns:", list(state_df.columns))

    tcol = next((c for c in ["time_ps", "frame_idx", "step"] if c in state_df.columns), state_df.columns[0])
    candidate_cols = [
        c for c in state_df.columns
        if c != tcol and c not in {"step", "frame_idx", "time_ps"}
        and pd.api.types.is_numeric_dtype(state_df[c])
    ]
    count_cols = [
        c for c in candidate_cols
        if c.lower() in {"sol", "na", "cl", "water"}
        or c.startswith("n_")
        or "count" in c.lower()
        or "core" in c.lower()
        or "bound" in c.lower()
    ]
    plot_cols = count_cols if count_cols else candidate_cols

    if plot_cols:
        fig, ax = plt.subplots(figsize=(8.2, 4.8))
        for c in plot_cols:
            ax.plot(state_df[tcol], state_df[c], label=c, lw=1.8)
        ax.set_title(f"Channel particle count per frame ({state_source})")
        ax.set_xlabel(tcol)
        ax.set_ylabel("Count")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, ncol=2)
        plt.tight_layout()
        savefig_and_show("channel_particle_count_timeseries", fig)


## 14) Drift in channel (summary table)
`drift_channel.csv` is displayed and exported as a summary table of channel drift metrics.

This dataset is summarized (not plotted) by design.


In [ ]:
if files["drift_channel"].exists():
    drift_df = read_csv(files["drift_channel"])
    print("drift_channel columns:", list(drift_df.columns))
    display(drift_df)
    save_table(drift_df, "drift_channel_summary.csv")


## 15) CSV coverage audit
Every CSV in `RUN_DIR` is classified as:
- `plotted`
- `summarized_only`
- `intentionally_skipped`
- `unhandled`

This guarantees that no run output file is silently ignored.


In [ ]:
plotted_stems = {
    "density_x",
    "density_z",
    "density_z_in_x_channel",
    "dipole_x",
    "dipole_z",
    "dipole_z_in_x_channel",
    "coord_x",
    "channel_count_xz",
    "state_z_channel",
    "gating_flux",
    "jump_msd",
    "msd_x_channel",
    "msd_z_channel",
    "vacf_y",
    "vacf_x_channel",
    "vacf_z_channel_raw",
}

summarized_only_stems = {
    "drift_channel",
}

intentionally_skipped_stems = set()

rows = []
for csv_path in RUN_CSV_FILES:
    stem = csv_stem_from_path(csv_path)

    if stem in plotted_stems:
        status = "plotted"
        note = ""
    elif stem in summarized_only_stems:
        status = "summarized_only"
        note = "shown as table"
    elif stem in intentionally_skipped_stems:
        status = "intentionally_skipped"
        note = "explicitly skipped"
    else:
        status = "unhandled"
        note = "no dedicated section"

    rows.append({
        "csv_file": csv_path.name,
        "stem": stem,
        "status": status,
        "note": note,
    })

coverage_df = pd.DataFrame(rows).sort_values(["status", "stem", "csv_file"]).reset_index(drop=True)
display(coverage_df)
save_table(coverage_df, "coverage_table.csv")

print("Coverage counts:")
print(coverage_df["status"].value_counts())

unhandled = coverage_df[coverage_df["status"] == "unhandled"]
if len(unhandled) > 0:
    print("\nUnhandled CSV files:")
    for fn in unhandled["csv_file"]:
        print("  -", fn)
